# ML-09 — Validation and Research Claim Audit

**Lane:** Refresh / Content Opportunity Scoring. **Decision supported:** which eligible pages should an editor inspect first for possible refresh?

This notebook audits the Week-5 model on the checked-in 30,000-row anonymized starter extract. Its target is `trend_direction == "down"`, a label derived from the latest 30 days versus the preceding 30 days. The extract does not provide a later independent outcome window, so every score below measures agreement with a trailing-window proxy, not future decline or the effect of refreshing a page.

The code uses only aggregate results and anonymous feature fields. It does not print client IDs, content IDs, URLs, or query text.

## 1. Two paper findings and methodology questions

**Finding A — Anatomy of growing content (CONFIRMED).** The report says growing pages averaged about 1,487 words and declining pages 1,481, while average age was 185 versus 228 days. This is a descriptive comparison of groups whose direction is defined from recent traffic change. It supports the narrow observation that these groups differed in age in this snapshot; it does not establish that age caused the difference. The report itself names age as a confounder in model comparisons. **Question:** Could the report show uncertainty intervals and a comparison stratified or adjusted for publication cohort, topic, and visibility, so readers can judge how much of the age gap remains within comparable pages?

**Finding B — unseen-brand growth model.** The report describes about 90% accuracy on unseen pages from known brands and 75% on brands not seen in training, and says it tested 20 approaches. The second result is more relevant to transfer across clients, but accuracy alone is hard to interpret without the positive-class base rate, class-specific errors, and a clear summary across the repeated splits. The report also identifies the ML pages as exploratory and the study as pattern-finding rather than causal proof. **Question:** Could the appendix report the target prevalence, balanced metrics (for example average precision and recall), variability across the 20 splits, and a later-time test, in addition to accuracy? That would make the transfer claim easier to assess.

These are questions about how to make the evidence easier to evaluate, not claims that the reported analysis is invalid. Source: FlyRank, *The State of AI-Driven SEO — Data Report, April 2026*, “The Anatomy of Growing Content,” “Which Pages Will Grow?,” and “Method & Limitations.”

In [1]:
# The paper audit above is a manual review of the linked report. This cell records its
# exact reference and checks that we are using Fiona's own notebook/repo context.
PAPER_URL = "https://state-of-seo-2026.flyrank.ai/"
NOTEBOOK_PATH = "work/notebooks/w06_validation_audit.ipynb"
print("Paper reviewed:", PAPER_URL)
print("Notebook path:", NOTEBOOK_PATH)
print("Review scope: two findings, label source, validation fit, and constructive questions")

Paper reviewed: https://state-of-seo-2026.flyrank.ai/
Notebook path: work/notebooks/w06_validation_audit.ipynb
Review scope: two findings, label source, validation fit, and constructive questions


## 2. Week-5 model under row-random and client-grouped splits

I re-run the same Logistic Regression pipeline and the same nine metadata features from Week 5. The row-random 75/25 split is shown as the before comparison. The client-grouped split holds all pages from eight clients out of training, testing transfer to unseen clients. Both use seed 42 and score the same ranking metrics. The target and eligible population remain unchanged (`impressions_90d >= 100`; decline proxy from `trend_direction`).

Precision@10 and Precision@50 are the share of the top-ranked pages that match the observed decline proxy. Average precision summarizes the ranked list; ROC AUC measures pairwise ordering. A single split is a useful audit, not a stable performance guarantee.

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score

DATA_REL = Path("data/raw/content_refresh_anonymized.csv")
repo_candidates = [Path.cwd(), Path.cwd() / "flyrank-ml-internship", Path("/content/flyrank-ml-internship")]
REPO_ROOT = next((p for p in repo_candidates if (p / DATA_REL).exists()), None)
if REPO_ROOT is None:
    import subprocess
    clone_path = Path("/tmp/flyrank-ml-internship")
    if not (clone_path / DATA_REL).exists():
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Fiona-ali/flyrank-ml-internship.git", str(clone_path)], check=True)
    REPO_ROOT = clone_path
DATA_PATH = REPO_ROOT / DATA_REL
df = pd.read_csv(DATA_PATH)
RAW_SHAPE = df.shape
assert RAW_SHAPE == (30_000, 44), f"Unexpected data shape: {RAW_SHAPE}"
required = {"client_id", "trend_direction", "impressions_90d", "search_volume", "competition", "cpc", "content_type", "main_intent", "word_count", "char_count", "content_age_days", "days_since_last_update"}
assert required.issubset(df.columns), f"Missing columns: {sorted(required - set(df.columns))}"
df["decline_proxy"] = df["trend_direction"].astype("string").str.lower().eq("down").astype(int)
eligible = df.loc[pd.to_numeric(df["impressions_90d"], errors="coerce") >= 100].copy().reset_index(drop=True)
NUMERIC_FEATURES = ["search_volume", "competition", "cpc", "word_count", "char_count", "content_age_days", "days_since_last_update"]
CATEGORICAL_FEATURES = ["content_type", "main_intent"]
FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES
FORBIDDEN = {"content_id", "client_id", "trend_direction", "trend_pct", "decline_proxy", "is_declining_label", "impressions_last_30d", "impressions_prev_30d", "clicks_last_30d", "clicks_prev_30d", "sessions_last_30d", "sessions_prev_30d", "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d", "users_90d", "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d", "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct"}
assert not (set(FEATURES) & FORBIDDEN)
X = eligible[FEATURES].copy()
y = eligible["decline_proxy"].to_numpy(dtype=int)
groups = eligible["client_id"].to_numpy()

def make_model():
    numeric_pipe = Pipeline([("impute", SimpleImputer(strategy="median", add_indicator=True)), ("scale", StandardScaler())])
    categorical_pipe = Pipeline([("impute", SimpleImputer(strategy="constant", fill_value="__MISSING__")), ("onehot", OneHotEncoder(handle_unknown="ignore"))])
    preprocess = ColumnTransformer([("numeric", numeric_pipe, NUMERIC_FEATURES), ("categorical", categorical_pipe, CATEGORICAL_FEATURES)])
    return Pipeline([("preprocess", preprocess), ("classifier", LogisticRegression(max_iter=1000, random_state=42))])

def metrics(y_true, scores):
    order = np.argsort(-np.asarray(scores), kind="mergesort")
    return {"Precision@10": float(y_true[order[:10]].mean()), "Precision@50": float(y_true[order[:50]].mean()), "Average precision": float(average_precision_score(y_true, scores)), "ROC AUC": float(roc_auc_score(y_true, scores))}

random_train, random_test = train_test_split(np.arange(len(eligible)), test_size=0.25, random_state=42, stratify=y)
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
group_train, group_test = next(gss.split(X, y, groups=groups))
assert set(groups[group_train]).isdisjoint(set(groups[group_test]))
summary = []
for name, tr, te in [("Row-random (Week-5 style)", random_train, random_test), ("Client-grouped (unseen clients)", group_train, group_test)]:
    m = make_model()
    m.fit(X.iloc[tr], y[tr])
    s = m.predict_proba(X.iloc[te])[:, 1]
    row = {"Split": name, "Train rows": len(tr), "Test rows": len(te), "Train clients": len(set(groups[tr])), "Test clients": len(set(groups[te])), "Test base rate": float(y[te].mean()), **metrics(y[te], s)}
    summary.append(row)
results = pd.DataFrame(summary)
print(f"Starter extract: {RAW_SHAPE[0]:,} rows × {RAW_SHAPE[1]} columns; eligible: {len(eligible):,}; clients: {df.client_id.nunique()}")
print(f"Model inputs ({len(FEATURES)}): {', '.join(FEATURES)}")
print("No client overlap in grouped split:", set(groups[group_train]).isdisjoint(set(groups[group_test])))
print(results.to_string(index=False, formatters={c:(lambda x: f"{x:.3f}") for c in ["Test base rate","Precision@10","Precision@50","Average precision","ROC AUC"]}))
print("Versions: pandas", pd.__version__, "scikit-learn", sklearn.__version__)

Starter extract: 30,000 rows × 44 columns; eligible: 22,006; clients: 32
Model inputs (9): search_volume, competition, cpc, word_count, char_count, content_age_days, days_since_last_update, content_type, main_intent
No client overlap in grouped split: True
                          Split  Train rows  Test rows  Train clients  Test clients Test base rate Precision@10 Precision@50 Average precision ROC AUC
      Row-random (Week-5 style)       16504       5502             30            28          0.598        0.700        0.860             0.686   0.621
Client-grouped (unseen clients)       17396       4610             22             8          0.577        0.600        0.720             0.632   0.571
Versions: pandas 2.2.3 scikit-learn 1.8.0


### Before/after reading

The table is the comparison from the actual rerun above. Interpret a change in average precision and top-K precision as a change in agreement with this observed proxy under a harder client-transfer split. If grouped results are lower, the row-random split was optimistic for unseen-client use; if they are similar or higher, that does not establish performance on future periods. The dataset has only 32 clients, so the held-out-client estimate depends on one particular group split. The next validation step should repeat the group split or, preferably, use a later time period and independently defined future outcome.

## 3. Leakage audit

The label is derived from `trend_direction`, which compares the last 30 days with the previous 30 days. The feature list contains only content metadata available before an editor decides what to review. The checks below verify that identifiers, label-derived fields, and traffic/performance columns are absent. IDs are used only to group the split, never as model inputs. Eligibility uses `impressions_90d >= 100`; that rule defines the evaluated population, so results apply only to that eligible subset. No feature uses the label period. Missingness is preserved with imputation indicators or an explicit category.

In [3]:
# Explicit feature leakage checks (all assertions must pass).
assert "client_id" not in FEATURES and "content_id" not in FEATURES
assert not (set(FEATURES) & FORBIDDEN)
assert "trend_direction" not in X.columns and "trend_pct" not in X.columns
assert all(c not in X.columns for c in ["impressions_last_30d", "impressions_prev_30d", "impressions_90d", "ctr", "avg_position"])
assert set(groups[group_train]).isdisjoint(set(groups[group_test]))
print("PASS: no IDs, target fields, or overlapping traffic/performance fields in X")
print("PASS: client_id is used only for grouped splitting")
print("PASS: training and test clients are disjoint")
print("Disclosure: evaluation is limited to rows with impressions_90d >= 100")
print("Disclosure: target is a trailing 30-day comparison proxy, not a future outcome")
print("Disclosure: one grouped split; 32 clients total; temporal validation is still needed")

PASS: no IDs, target fields, or overlapping traffic/performance fields in X
PASS: client_id is used only for grouped splitting
PASS: training and test clients are disjoint
Disclosure: evaluation is limited to rows with impressions_90d >= 100
Disclosure: target is a trailing 30-day comparison proxy, not a future outcome
Disclosure: one grouped split; 32 clients total; temporal validation is still needed


## 4. Claim rewrite

**Bold claim from the Week-5 write-up:** “The logistic model improved ranking over the Week-4 rule.”

**Safer rewrite:** “On one fixed client-held-out split of the 30,000-row teaching extract, the Week-5 logistic model reached average precision 0.632 and Precision@50 0.720 against the trailing decline proxy (test base rate 0.577). These are directional, sample-specific results for the eligible pages; they do not establish future predictive performance or show that refreshing a page causes improvement. Use scores only to prioritize editor review, then validate on later data with an independently defined outcome.”

This wording reports what was measured, names the validation scope and proxy, and limits the action to decision support.

## Self-check

- [x] Two paper findings identify the label/evidence and ask constructive methodology questions.
- [x] Week-5 Logistic Regression re-run under row-random and client-grouped splits on the same starter extract.
- [x] Actual row counts, base rates, ranking metrics, and library versions printed by the code.
- [x] Leakage assertions pass; IDs are used only for grouping.
- [x] Claims use careful language and describe the proxy and limitations.
- [x] No client names, IDs, URLs, or private queries are printed.
- [x] Run top-to-bottom in Colab after cloning Fiona’s repository; the public starter data remains unchanged.
